### Random Forest

Random Forest is an ensemble learning method that constructs multiple decision trees during training and outputs the mode of the classes (classification) or mean prediction (regression) of the individual trees. It is widely used for its accuracy and ability to handle large datasets with higher dimensionality.

In [ ]:
from math import log2
import pandas as pd

First, we find the entropy of the target variable. The entropy is calculated using the formula:
$$H(Y) = -\sum_{i=1}^{n} p_i \log_2(p_i)$$
where $p_i$ is the probability of class $i$ in the target variable. The entropy gives us a measure of the impurity or disorder in the target variable, and it is used to determine the best feature to split the data at each node of the decision tree. The feature that results in the highest information gain (reduction in entropy) is chosen for the split.

The second step is to calculate the entropy of the feature. The entropy of a feature is calculated by considering the distribution of the target variable for each value of the feature. The formula for calculating the entropy of a feature is:
$$H(X) = \sum_{j=1}^{m} p_j H(Y|X_j)$$
where $p_j$ is the probability of the feature taking the value $j$, and $H(Y|X_j)$ is the conditional entropy of the target variable given that the feature takes the value $j$. The conditional entropy is calculated using the same formula as the entropy of the target variable, but it is calculated for each subset of the data corresponding to the value of the feature.

Finally, we calculate the information gain for each feature using the formula:
$$IG(Y, X) = H(Y) - H(X)$$
The feature with the highest information gain is selected for the split at each node of the decision tree. This process is repeated recursively until a stopping criterion is met, such as a maximum depth of the tree or a minimum number of samples required to split a node. The resulting decision trees are then combined to form the random forest, which makes predictions based on the majority vote (for classification) or average (for regression) of the individual trees.

In [ ]:
# Entropy of target
def entropy_target(target):
    p_list = []
    for val in target.unique():
        p = target.value_counts().get(val, 0) / len(target)
        p_list.append(p)
    return sum(-p * log2(p) for p in p_list if p > 0)


# Entropy of feature 
def entropy_feature(feature, target):
    df = pd.concat([feature, target], axis=1)
    entropy_list = []

    for f_val in feature.unique():
        p_list = []
        num_class = feature.value_counts().get(f_val, 0)

        for t_val in target.unique():
            sub_df = df[(df[feature.name] == f_val) &
                        (df[target.name] == t_val)]
            p = len(sub_df) / num_class if num_class != 0 else 0
            p_list.append(p)

        entropy = sum(-p * log2(p) for p in p_list if p > 0)
        entropy_list.append(entropy)

    return entropy_list


# Information Gain
def information_gain(x, y):
    e_target = entropy_target(y)
    results = {}

    for col in x.columns:
        feature = x[col]
        unique_counts = [feature.value_counts().get(v, 0) for v in feature.unique()]
        entropy_list = entropy_feature(feature, y)

        # Weighted entropy
        weight = sum((unique_counts[i] / len(x)) * entropy_list[i]
                     for i in range(len(unique_counts)))

        ig = e_target - weight
        results[col] = {"information_gain": float(ig)}

    best_feature = max(results, key=lambda k: results[k]["information_gain"])


    return results, best_feature

This function calculates the entropy of the target variable, which is a measure of the impurity or disorder in the target variable. It takes a list of target values as input and returns the entropy value. The function first calculates the probability of each class in the target variable and then uses these probabilities to calculate the entropy using the formula mentioned above.

In [877]:
def recursive_split(X, y, best_feature):
    next_best = {}

    for value in X[best_feature].unique():
        print(f"\nSubset: {best_feature} = {value}")

        sub_X_full = X[X[best_feature] == value]
        sub_y = y[X[best_feature] == value]

        # Concatenate
        print(pd.concat([sub_X_full.drop(columns=[best_feature]), sub_y], axis=1))

        # If pure → store leaf
        if len(sub_y.unique()) == 1:
            print("Pure subset")
            next_best[value] = {"type": "leaf", "class": sub_y.iloc[0]}
            continue

        # Otherwise compute next best feature
        sub_X = sub_X_full.drop(columns=[best_feature])
        sub_info, sub_best_feature = information_gain(sub_X, sub_y)
        for feature, info in sub_info.items():
            print(f"{feature}: {info}")
        print("+ Best Feature:", sub_best_feature)

        next_best[value] = {"type": "node",
                            "information_gain": sub_info,
                            "value": value,
                            "best_feature": sub_best_feature}

    return next_best

In [878]:
# Now we filter the dataset based on the root feature and the best feature and check for its impurity
def process_next_level(X, y, best_feature, next_best_features):
    all_next_levels = {}
    for value, info in next_best_features.items():
        print(f"\nProcessing branch: {best_feature} = {value}")
        # Only continue if impure
        if info["type"] == "node":
            sub_best_feature = info["best_feature"]
            # Filter using root feature
            X_sub = X[X[best_feature] == value].drop(columns=[best_feature])
            y_sub = y[X[best_feature] == value]
            print(f"→ Going deeper using feature: {sub_best_feature}")
            # Call recursive_split again
            next_level = recursive_split(X_sub, y_sub, sub_best_feature)
            # Store result (optional but useful)
            all_next_levels[value] = next_level
        else:
            print("→ Leaf node, stopping.")
            all_next_levels[value] = info

    return all_next_levels

Apply the same process to calculate the entropy of the feature and the information gain for each feature. The feature with the highest information gain is selected for the split at each node of the decision tree. This process is repeated recursively until a stopping criterion is met, such as a maximum depth of the tree or a minimum number of samples required to split a node. The resulting decision trees are then combined to form the random forest, which makes predictions based on the majority vote (for classification) or average (for regression) of the individual trees.

In [879]:
sub1 = {
    "Age": ["≤30", ">40", ">40", "≤30", "≤30", "31…40", "≤30", ">40", "31…40", "31…40"],
    "Income": ["high", "low", "low", "low", "high", "medium", "high", "medium", "low", "high"],
    "Student": ["no", "yes", "yes", "yes", "no", "no", "no", "no", "yes", "no"],
    "Credit_Rating": ["Excellent", "Fair", "Fair", "Fair", "Fair", "Excellent", "Excellent", "Excellent", "Excellent", "Fair"],
    "Buy_XBOX": ["no", "yes", "yes", "yes", "no", "yes", "no", "no", "yes", "yes"]
}

X = pd.DataFrame(sub1)
y = X.pop("Buy_XBOX")
results, best_feature = information_gain(X, y)
# Iterate through the result dictionary
for feature, info in results.items():
      print(f"{feature}: {info}")
print("+ Best Feature:", best_feature, "\n")

next_best_features = recursive_split(X, y, best_feature)
all_next_levels = process_next_level(X, y, best_feature, next_best_features)

Age: {'information_gain': 0.3709505944546686}
Income: {'information_gain': 0.4464393446710154}
Student: {'information_gain': 0.4199730940219749}
Credit_Rating: {'information_gain': 0.12451124978365313}
+ Best Feature: Income 


Subset: Income = high
     Age Student Credit_Rating Buy_XBOX
0    ≤30      no     Excellent       no
4    ≤30      no          Fair       no
6    ≤30      no     Excellent       no
9  31…40      no          Fair      yes
Age: {'information_gain': 0.8112781244591328}
Student: {'information_gain': 0.0}
Credit_Rating: {'information_gain': 0.31127812445913283}
+ Best Feature: Age

Subset: Income = low
     Age Student Credit_Rating Buy_XBOX
1    >40     yes          Fair      yes
2    >40     yes          Fair      yes
3    ≤30     yes          Fair      yes
8  31…40     yes     Excellent      yes
Pure subset

Subset: Income = medium
     Age Student Credit_Rating Buy_XBOX
5  31…40      no     Excellent      yes
7    >40      no     Excellent       no
Age: {'inform

In [880]:
# Repeat for these twos
sub2 = {
    "Age": [">40", "≤30", "≤30", ">40", ">40", "≤30", "31…40", ">40", "≤30", "≤30"],
    "Income": ["medium", "medium", "medium", "medium", "low", "medium", "high", "medium", "high", "low"],
    "Student": ["no", "no", "no", "yes", "yes", "yes", "yes", "no", "no", "yes"],
    "Credit_Rating": ["Fair", "Fair", "Fair", "Fair", "Excellent", "Excellent", "Fair", "Fair", "Excellent", "Fair"],
    "Buy_XBOX": ["yes", "no", "no", "yes", "no", "yes", "yes", "yes", "no", "yes"]
}

X = pd.DataFrame(sub2)
y = X.pop("Buy_XBOX")
results, best_feature = information_gain(X, y)
# Iterate through the result dictionary
for feature, info in results.items():
      print(f"{feature}: {info}")
print("+ Best Feature:", best_feature, "\n")

next_best_features = recursive_split(X, y, best_feature)
all_next_levels = process_next_level(X, y, best_feature, next_best_features)

Age: {'information_gain': 0.1609640474436811}
Income: {'information_gain': 0.01997309402197489}
Student: {'information_gain': 0.12451124978365313}
Credit_Rating: {'information_gain': 0.0912774462416801}
+ Best Feature: Age 


Subset: Age = >40
   Income Student Credit_Rating Buy_XBOX
0  medium      no          Fair      yes
3  medium     yes          Fair      yes
4     low     yes     Excellent       no
7  medium      no          Fair      yes
Income: {'information_gain': 0.8112781244591328}
Student: {'information_gain': 0.31127812445913283}
Credit_Rating: {'information_gain': 0.8112781244591328}
+ Best Feature: Income

Subset: Age = ≤30
   Income Student Credit_Rating Buy_XBOX
1  medium      no          Fair       no
2  medium      no          Fair       no
5  medium     yes     Excellent      yes
8    high      no     Excellent       no
9     low     yes          Fair      yes
Income: {'information_gain': 0.4199730940219749}
Student: {'information_gain': 0.9709505944546686}
Credit_R

In [881]:
sub3 = {
    "Age": ["≤30", "31…40", "31…40", ">40", ">40", ">40", "31…40", ">40", ">40", "31…40"],
    "Income": ["high", "high", "high", "low", "low", "low", "medium", "medium", "medium", "low"],
    "Student": ["no", "no", "no", "yes", "yes", "yes", "no", "yes", "no", "yes"],
    "Credit_Rating": ["Fair", "Fair", "Fair", "Fair", "Excellent", "Excellent", "Excellent", "Fair", "Excellent", "Excellent"],
    "Buy_XBOX": ["no", "yes", "yes", "yes", "no", "no", "yes", "yes", "no", "yes"]
}

X = pd.DataFrame(sub3)
y = X.pop("Buy_XBOX")
results, best_feature = information_gain(X, y)
# Iterate through the result dictionary
for feature, info in results.items():
      print(f"{feature}: {info}")
print("+ Best Feature:", best_feature, "\n")

next_best_features = recursive_split(X, y, best_feature)
all_next_levels = process_next_level(X, y, best_feature, next_best_features)

Age: {'information_gain': 0.4854752972273343}
Income: {'information_gain': 0.01997309402197489}
Student: {'information_gain': 0.0}
Credit_Rating: {'information_gain': 0.12451124978365313}
+ Best Feature: Age 


Subset: Age = ≤30
  Income Student Credit_Rating Buy_XBOX
0   high      no          Fair       no
Pure subset

Subset: Age = 31…40
   Income Student Credit_Rating Buy_XBOX
1    high      no          Fair      yes
2    high      no          Fair      yes
6  medium      no     Excellent      yes
9     low     yes     Excellent      yes
Pure subset

Subset: Age = >40
   Income Student Credit_Rating Buy_XBOX
3     low     yes          Fair      yes
4     low     yes     Excellent       no
5     low     yes     Excellent       no
7  medium     yes          Fair      yes
8  medium      no     Excellent       no
Income: {'information_gain': 0.01997309402197489}
Student: {'information_gain': 0.17095059445466854}
Credit_Rating: {'information_gain': 0.9709505944546686}
+ Best Feature: Cre